# 04 · Readout and fine-tuning

**pragmatiq series, notebook 4 of 4 after the [tour](00_blueprint_v2_tour.ipynb). Slot 4: which state becomes the
embedding, and how a downstream task uses it.**

The tour (notebook 00) fine-tuned the checkpoint on five tasks and scored
each against XGBoost on raw features. This notebook opens that step up on one
task, `churn_6m`:

1. **Read out** the embedding, the decoder state after a row's last event,
   and look at the embedding space.
2. **Score the baseline**: XGBoost on the row's raw features, with no history.
3. **Fine-tune** the pretrained network with LoRA, and watch early stopping
   choose the step.
4. **Count** what LoRA trains, and what its rank changes.

About 5 minutes on the lab GPU with the default 4,000-user sample. The sample,
split, and seed are the tour's, so this notebook's churn scores match the
tour's churn row.

> **Smoke mode.** CI runs this notebook with `TFM_NOTEBOOK_SMOKE=1`: a
> generated 120-user book, an untrained model, and CPU only. Every cell runs,
> but the numbers and plots are meaningless. The first code cell prints which
> mode you are in.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tfm.runtime.lab import fetch_lab_assets, lab_tasks, load_lab_model, prepare_lab_data
from tfm.training.reproducibility import seed_everything

GREEN, DARK, GREY = "#76B900", "#1A1A1A", "#8C8C8C"
PALETTE = [GREEN, "#0074DF", "#F2A900", "#5D1682", "#E52020", DARK, GREY, "#00A3A3"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

assets = fetch_lab_assets("04")
config = assets.config
seed_everything(config.seed)
print(assets.describe())

In [ ]:
from tfm.runtime.evaluate import fraud_entity_split, fraud_task

data = prepare_lab_data(assets)
model = load_lab_model(assets, data)
random_model = load_lab_model(assets, data, pretrained=False)
profiles = {p.entity_id: p for p in data.profiles}

tasks = {"fraud_user": fraud_task(data, fraud_entity_split(data, config.seed)),
         **lab_tasks(assets, data, ("fraud_txn", "churn_6m", "default_12m", "ltv_positive"))}
TASK = "churn_6m"  # any of the five; see the exercises
task = tasks[TASK]
METRIC = "roc_auc" if TASK == "ltv_positive" else "pr_auc"
pd.DataFrame({name: t.summary() for name, t in tasks.items()}).T

The five tasks from notebook 01:

| Task | One row is… | Label |
|---|---|---|
| `fraud_user` | a user's history at the end of a side's label window | 1 if a theft day was spliced into it |
| `fraud_txn` | one labelled card transaction, with the history up to and including it | 1 if it was copied from another user |
| `churn_6m` | a user's history at 2025-01-01 | 1 if the user goes quiet in the next six months |
| `default_12m` | a user's history at 2024-07-01 | 1 if the user defaults within a year |
| `ltv_positive` | a user's history at 2025-01-01 | 1 if the next six months are profitable |

This notebook follows `churn_6m`: about 2,200 train-side users and 940
evaluation users, 13% of whom churn. Set `TASK` above to follow another.

## The readout

**The embedding** (`last_token`) is the decoder state after the row's last
event: 256 numbers that summarize the profile and the history up to the
row's cutoff. It uses only events up to that cutoff, and reading it changes
no weight. A contract test checks that dropping later events leaves it
unchanged.

In [ ]:
from tfm.inference.embedding import extract_embeddings

started = time.perf_counter()
embeddings = {name: extract_embeddings(m, task.eval_batch, device=assets.device, minibatch_size=64).vectors
              for name, m in {"pretrained": model, "random init": random_model}.items()}
print(f"{embeddings['pretrained'].shape[1]} numbers per row for {len(task.y_eval):,} {TASK} rows; "
      f"{time.perf_counter() - started:.0f} s")

## The embedding space

UMAP projects each evaluation row's 256-number embedding to two dimensions. Each
column colours the same points by a different attribute. The top row is the
pretrained model; the bottom row is the same network untrained. Neither
model ever saw these attributes as labels. The profile's country and income
band are inputs; the task's label is not.

In [ ]:
from tfm.evaluation.representation.projection import project_2d

ids = task.eval_batch.entity_ids
colour_by = {
    "country": [profiles[e].attributes.get("country", "?") for e in ids],
    "income band": [profiles[e].attributes.get("income_band", "?") for e in ids],
    f"{TASK} label": ["positive" if y else "negative" for y in task.y_eval],
}
fig, axes = plt.subplots(2, 3, figsize=(14, 8.4))
for row, model_name in enumerate(("pretrained", "random init")):
    xy = project_2d(embeddings[model_name], method="umap", seed=config.seed)
    for col, (title, values) in enumerate(colour_by.items()):
        ax = axes[row, col]
        labels = pd.Series(values)
        colours = {"negative": GREY, "positive": "#E52020"} if col == 2 else {}
        for i, value in enumerate(labels.value_counts().index[:7]):
            chosen = (labels == value).to_numpy()
            ax.scatter(xy[chosen, 0], xy[chosen, 1], s=5, alpha=0.6,
                       color=colours.get(value, PALETTE[i % len(PALETTE)]), label=value)
        ax.set_title(f"{model_name}: {title}")
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.legend(fontsize=7, markerscale=2, loc="best")
plt.tight_layout()
plt.show()

The pretrained embedding sorts users by country, one of the profile inputs,
into separate islands. Income band, another input, shows little structure.
Churners are scattered through every island rather than gathered in one,
so no single direction of the embedding says "will churn": the classifier
has to find that signal. The untrained network gives a few tight clumps
that only partly follow country and mix income bands and churners
throughout.

The embedding is a summary, not an answer. To turn it into a churn
forecast, the rest of this notebook adapts the whole network to the task,
and checks the result against a model that never saw the history.

## The baseline: XGBoost on raw features

The baseline is Blueprint v1's raw arm: XGBoost on the row's last event's
own fields and the user's profile, with **no history**. Its settings are
fixed (`LAB_XGBOOST_PARAMS` in `tfm.runtime.lab`), shared by every task, and
set to the median of what earlier hyperparameter searches picked. Early
stopping on held-out train-side users sets the number of trees; nothing else
is tuned here.

The metric is PR-AUC, the area under the precision-recall curve. It suits
rare positives; a random guess scores the positive rate, 0.13 for churn.
(`ltv_positive` is the one task scored by ROC-AUC: about 71% of users are
positive, so every model's PR-AUC starts near 0.9.)

In [ ]:
from tfm.evaluation.tabular import evaluate_raw_embedding_arms
from tfm.runtime.lab import LAB_XGBOOST_PARAMS

baselines = config.evaluation.tabular_baselines
arms, _ = evaluate_raw_embedding_arms(
    task.train_sequences, task.y_train, task.eval_sequences, task.y_eval,
    profiles=profiles, embedding_views={},
    xgboost_params=LAB_XGBOOST_PARAMS, seed=config.seed, top_k=baselines.top_k,
    coded_keys=tuple(config.features.categorical_keys), n_jobs=baselines.n_jobs,
    k=config.evaluation.precision_at_k,
)
raw = arms["xgboost_raw"]
print(f"{TASK}, XGBoost on raw features: PR-AUC {raw['pr_auc']:.3f}, ROC-AUC {raw['roc_auc']:.3f} "
      f"(random ranking: PR-AUC {task.y_eval.mean():.3f})")

## Fine-tuning with LoRA

A frozen embedding leaves the encoder untouched. Fine-tuning adapts it to
the task. **LoRA** (low-rank adaptation) freezes every pretrained weight and
learns a small update `B·A` of rank 16 beside chosen weight matrices: here
the attention projections and both feed-forward layers of every block in
all three stacks. A linear head reads the embedding. The fine-tune works on
a copy of the checkpoint, so `model` itself never changes.

The settings come from `fraud_task.finetune` in the checkpoint's
`config.yaml`. They hold out 20% of the train-side users for validation,
check the validation loss every 50 steps, and stop after 4 checks without
improvement, restoring the best weights. The evaluation side is never seen.

In [ ]:
from tfm.adaptation.finetune import finetune_binary
from tfm.runtime.evaluate import finetune_spec

spec = finetune_spec(config)
started = time.perf_counter()
tuned, manifest = finetune_binary(
    model, task.train_batch, task.eval_batch, task.y_train, task.y_eval, spec,
    device=assets.device, seed=config.seed, k=config.evaluation.precision_at_k,
    precision=config.training.precision,
)
print(f"LoRA rank {spec.lora.rank} on {', '.join(spec.lora.targets)}: "
      f"{manifest['n_trainable_params']:,} trainable parameters; {len(task.y_train):,} train-side rows "
      f"({int(task.y_train.sum()):,} positive); stopped at step {manifest['steps']}, "
      f"best {manifest['best_step']}; {time.perf_counter() - started:.0f} s")

In [ ]:
curve = np.array(manifest["validation_curve"], dtype=float).reshape(-1, 2)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(curve[:, 0], curve[:, 1], "o-", color=GREEN)
ax.axvline(manifest["best_step"], color=DARK, ls="--", label=f"best step {manifest['best_step']}, restored")
ax.set_xlabel("fine-tuning step")
ax.set_ylabel("validation loss")
ax.set_title(f"LoRA on {TASK}: early stopping on held-out train-side users")
ax.legend()
plt.tight_layout()
plt.show()

The validation loss falls at first and then turns up, once the adapters
start to memorize the training rows; early stopping keeps the best
checkpoint. Every task stops within a few hundred steps, so a fine-tune
takes one to two minutes.

## Raw features against the fine-tuned TFM

In [ ]:
result = pd.DataFrame({"XGBoost, raw": raw, "TFM, LoRA": tuned}).T[["pr_auc", "roc_auc"]]
print(f"{TASK}: {METRIC.replace('_', '-').upper()} lift {tuned[METRIC] / raw[METRIC] - 1:+.0%}")
result.round(3)

The fine-tuned TFM ranks churners well above the raw-feature model, on both
metrics. The baseline sees only the latest event and the profile, by
design; the TFM reads the last 128 events, so the lift is what the history
adds. This is one row of the tour's five-task table, with the same numbers.

## What LoRA trains

LoRA's cost is set by its rank: each adapted `d_in × d_out` layer gains
`rank × (d_in + d_out)` trainable numbers, and everything else stays frozen.
Counting them needs no training:

In [ ]:
import copy

from tfm.training.lora import LoRA, apply_lora

counts = {}
for rank in (4, 8, 16, 32):
    stats = apply_lora(copy.deepcopy(model), LoRA(rank=rank, alpha=float(rank), targets=spec.lora.targets))
    counts[rank] = {"adapted layers": stats.n_adapted, "trainable parameters": f"{stats.n_trainable:,}",
                    "share of the network": f"{stats.n_trainable / (stats.n_trainable + stats.n_frozen):.1%}"}
pd.DataFrame(counts).T.rename_axis("LoRA rank")

At rank 16, the lab's setting, the adapters are about 5% of the network.
Rank 8 halves that. The lab's final sweep tried both and kept 16, which
lifted the downstream tasks more on the 3,000-step checkpoint; exercise 2
lets you compare them on this task.

## Other adaptation options

`fraud_task.finetune` chooses what moves and what reads the embedding:

- `method: lora` (low-rank adapters) or `method: full` (every weight);
- `rank` and `alpha` set the adapters' size and scale (16 and 16.0 here);
- `head: linear`, or `head: joint_fusion`, nuFormer's cross network over the
  embedding and periodic-embedded raw features, trained end to end.

## What these numbers do not show

- Anything about real customers. The book is synthetic: a toy problem,
  built to show how a TFM is pretrained, adapted, and tested on downstream
  tasks.
- A range. One pretraining seed, one architecture, one 4,000-user sample,
  one customer split.
- A tuned baseline. One fixed classifier setting; a search per task moves
  single scores by a few points.

No fraud-detection, credit, or production claim follows from this notebook.

### Exercises

1. Set `TASK = "fraud_user"` (or any of the five) and rerun from the setup cell. Does early stopping pick a later or earlier step than for churn, and does your score match that task's row in the tour?
2. Halve the rank: run `spec = dataclasses.replace(spec, lora=dataclasses.replace(spec.lora, rank=8, alpha=8.0))` (after `import dataclasses`) and rerun only the `finetune_binary` call. How much does the score move for half the trainable parameters?
3. Let the fine-tuned model see the raw fields too: `spec = dataclasses.replace(spec, head="joint_fusion")`. Does it beat the linear head?
4. Sample more users with `fetch_lab_assets("04", n_users=10_000)` (about 36 GB of host memory). Does the lift on `default_12m`, the task with the fewest positives, move more than churn's?

## Where this lives in the kit

This notebook explains; the recipe runs. Every cell calls `src/tfm`, so the
same code runs in [`recipes/blueprint_v2`](../../recipes/blueprint_v2), and no
reusable logic lives here.

Readout: `tfm.inference.embedding.extract_embeddings`. Baseline:
`tfm.evaluation.tabular.evaluate_raw_embedding_arms`. Tasks:
`tfm.runtime.lab.lab_tasks` and `tfm.runtime.evaluate.fraud_task`.
Fine-tuning: `tfm.adaptation.finetune.finetune_binary`, with the settings
`tfm.runtime.evaluate.finetune_spec` reads from `fraud_task.finetune`, and
`tfm.training.lora.apply_lora`.

Next: [05 · Scaling up](05_scaling_up.ipynb), the configs, commands, and recorded runs behind the checkpoint.